# Uniswap V3 data download and strategy analysis
This notebook downloads Uniswap v3 data from The Graph and an RPC, then performs cleaning, feature engineering, metrics computation, and strategy backtesting.

## 1. Install dependencies (pip)
If you need pinned versions, manage them in `requirements.txt` or Poetry. This cell records the current environment package versions.

In [ ]:
# %pip install pandas numpy requests web3 pyarrow matplotlib seaborn plotly python-dotenv gql[requests] tqdm

import importlib.metadata as md

packages = [
    "pandas",
    "numpy",
    "requests",
    "web3",
    "pyarrow",
    "matplotlib",
    "seaborn",
    "plotly",
    "python-dotenv",
    "gql",
    "tqdm",
]

versions = {pkg: md.version(pkg) for pkg in packages}
versions

## 2. Configuration & environment variables
Set `GRAPH_API_KEY`, `SUBGRAPH_ID` (or provide `GRAPH_ENDPOINT`), and `RPC_URL` in your `.env` file.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

root_dir = Path("..")
load_dotenv(root_dir / "app.env", override=True)
load_dotenv(root_dir / ".env", override=True)

In [ ]:
GRAPH_API_KEY = os.getenv("GRAPH_API_KEY", "")
SUBGRAPH_API_BASE = os.getenv("SUBGRAPH_API_BASE", "")
ARBITRUM_SUBGRAPH_ID = os.getenv("ARBITRUM_SUBGRAPH_ID", "")
UNISWAP_V3_ARBITRUM_SUBGRAPH_ID = os.getenv("UNISWAP_V3_ARBITRUM_SUBGRAPH_ID", "")

ARBITRUM_ENDPOINT = f"{SUBGRAPH_API_BASE}/{ARBITRUM_SUBGRAPH_ID}"
UNISWAP_V3_ARBITRUM_ENDPOINT = f"{SUBGRAPH_API_BASE}/{UNISWAP_V3_ARBITRUM_SUBGRAPH_ID}"

if GRAPH_API_KEY is None or GRAPH_API_KEY == "":
    print("Warning: GRAPH_API_KEY is not set in environment variables.")

ARBITRUM_ENDPOINT, UNISWAP_V3_ARBITRUM_ENDPOINT

In [ ]:
ALCHEMY_API_KEY = os.getenv("ALCHEMY_API_KEY", "")
RPC_URL = os.getenv("ALCHEMY_ARBITRUM_URL", "")

if ALCHEMY_API_KEY is None or ALCHEMY_API_KEY == "":
    print("Warning: ALCHEMY_API_KEY is not set in environment variables.")

RPC_URL if RPC_URL else ""

## 3. Connect and verify data sources (The Graph / RPC)
Verify that the GraphQL and RPC endpoints are reachable.

In [ ]:
from gql import gql, Client
from gql.transport.requests import RequestsHTTPTransport

headers = {"Authorization": f"Bearer {GRAPH_API_KEY}"} if GRAPH_API_KEY else {}

# Set up GraphQL client for Arbitrum subgraph
arbitrum_transport = RequestsHTTPTransport(url=ARBITRUM_ENDPOINT, verify=True, retries=3, headers=headers)
arbitrum_client = Client(transport=arbitrum_transport, fetch_schema_from_transport=False)

# Set up GraphQL client for Uniswap V3 Arbitrum subgraph
uniswap_v3_arbitrum_transport = RequestsHTTPTransport(url=UNISWAP_V3_ARBITRUM_ENDPOINT, verify=True, retries=3, headers=headers)
uniswap_v3_arbitrum_client = Client(transport=uniswap_v3_arbitrum_transport, fetch_schema_from_transport=False)

In [ ]:
test_query = gql("""
{
  _meta {
    block { number }
  }
}
""")

test_meta = arbitrum_client.execute(test_query)
test_meta

In [ ]:
from web3 import Web3

# Add api key to headers
headers = {"Authorization": f"Bearer {ALCHEMY_API_KEY}"} if ALCHEMY_API_KEY else {}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"headers": headers}))

if RPC_URL:
    assert w3.is_connected(), "RPC connection failed"
    chain_id = w3.eth.chain_id
else:
    chain_id = None

chain_id

## 4. Download Uniswap v3 pools and trades
Fetch swaps, ticks, and liquidity events by pool address and time range and cache results.

### Subgraph information

* Uniswap V3 Arbitrum 4.0.1_1.5.3 created by Messari

https://thegraph.com/explorer/subgraphs/FQ6JYszEKApsBpAmiHesRsd9Ygc6mzmpNRANeVQFYoVX?view=Query&chain=arbitrum-one

* Query URL

https://gateway.thegraph.com/api/subgraphs/id/FQ6JYszEKApsBpAmiHesRsd9Ygc6mzmpNRANeVQFYoVX


### 4.1 gpl

#### Fetch subgraph schemas

* Get the schema of all the entries.

In [ ]:
schema_query = gql("""
{
  __type(name: "Query") {
    fields {
      name
      args {
        name
        type {
          name
          kind
        }
      }
    }
  }
}
"""
)

schema_result = uniswap_v3_arbitrum_client.execute(schema_query)
display(schema_result)


* Get the schema of entry for these: 'swaps', 'liquidityPools', 'ticks', 'usageMetricsDailySnapshots' and 'liquidityPoolHourlySnapshots'.

In [ ]:
schema_swap = gql("""
{
  __type(name: "Swap") {
    name
    fields {
      name
      description
      type {
        name
        kind
        ofType {
          name
          kind
        }
      }
    }
  }
}
"""
)

schema_swap_result = uniswap_v3_arbitrum_client.execute(schema_swap)
display(schema_swap_result)

In [ ]:
schema_liquidityPool = gql("""
{
  __type(name: "LiquidityPool") {
    name
    fields {
      name
      description
      type {
        name
        kind
        ofType {
          name
          kind
        }
      }
    }
  }
}
"""
)

schema_liquidityPool_result = uniswap_v3_arbitrum_client.execute(schema_liquidityPool)
display(schema_liquidityPool_result)

In [ ]:
schema_tick = gql("""
{
  __type(name: "Tick") {
    name
    fields {
      name
      description
      type {
        name
        kind
        ofType {
          name
          kind
        }
      }
    }
  }
}
"""
)

schema_tick_result = uniswap_v3_arbitrum_client.execute(schema_tick)
display(schema_tick_result)

In [ ]:
schema_usageMetricsDailySnapshot = gql("""
{
  __type(name: "UsageMetricsDailySnapshot") {
    name
    fields {
      name
      description
      type {
        name
        kind
        ofType {
          name
          kind
        }
      }
    }
  }
}
"""
)

schema_usageMetricsDailySnapshot_result = uniswap_v3_arbitrum_client.execute(schema_usageMetricsDailySnapshot)
display(schema_usageMetricsDailySnapshot_result)

In [ ]:
schema_liquidityPoolHourlySnapshot = gql("""
{
  __type(name: "LiquidityPoolHourlySnapshot") {
    name
    fields {
      name
      description
      type {
        name
        kind
        ofType {
          name
          kind
        }
      }
    }
  }
}
"""
)

schema_liquidityPoolHourlySnapshot_result = uniswap_v3_arbitrum_client.execute(schema_liquidityPoolHourlySnapshot)
display(schema_liquidityPoolHourlySnapshot_result)

#### Get data

* Get pool addresses on Uniswap

https://app.uniswap.org/explore/pools/arbitrum

* ETH/USDC

0xC6962004f452bE9203591991D15f6b388e09E8D0

* ETH/USDT

0x641C00A822e8b671738d32a431a4Fb6074E5c79d

In [ ]:
ETH_USDC_POOL_ADDRESS = os.getenv("UNISWAP_V3_ARBITRUM_ETH_USDC_POOL_ADDRESS", "0xC6962004f452bE9203591991D15f6b388e09E8D0")
ETH_USDT_POOL_ADDRESS = os.getenv("UNISWAP_V3_ARBITRUM_ETH_USDT_POOL_ADDRESS", "0x641C00A822e8b671738d32a431a4Fb6074E5c79d")

ETH_USDC_POOL_ADDRESS, ETH_USDT_POOL_ADDRESS

Download data from liquidityPoolHourlySnapshots.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
from datetime import datetime, timezone


# Small sample settings
DATA_DIR = (root_dir / "var" / "raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)
POOL_ADDRESS = ETH_USDC_POOL_ADDRESS.lower()
SAMPLE_SIZE = int(os.getenv("SAMPLE_SIZE", "24"))  # fetch only a small number of rows

In [ ]:
HOURLY_SNAPSHOT_QUERY = gql("""
query($pool: String!, $first: Int!) {
  liquidityPoolHourlySnapshots(
    first: $first,
    orderBy: timestamp,
    orderDirection: desc,
    where: { pool: $pool }
  ) {
    id
    hour
    timestamp
    blockNumber
    tick
    totalLiquidity
    totalLiquidityUSD
    activeLiquidity
    activeLiquidityUSD
    totalValueLockedUSD
    hourlyVolumeUSD
    hourlySupplySideRevenueUSD
    hourlyProtocolSideRevenueUSD
    hourlyTotalRevenueUSD
    hourlySwapCount
  }
}
""")

# Execute a small sample query against the Uniswap V3 Arbitrum subgraph
resp = uniswap_v3_arbitrum_client.execute(
    HOURLY_SNAPSHOT_QUERY,
    variable_values={"pool": POOL_ADDRESS, "first": SAMPLE_SIZE},
)
rows = resp.get("liquidityPoolHourlySnapshots", [])

snapshots_df = pd.DataFrame(rows)
if not snapshots_df.empty:
    snapshots_df["timestamp"] = snapshots_df["timestamp"].astype(int)
    snapshots_df["datetime"] = pd.to_datetime(snapshots_df["timestamp"], unit="s", utc=True)
    numeric_cols = [
        "tick",
        "totalLiquidity",
        "totalLiquidityUSD",
        "activeLiquidity",
        "activeLiquidityUSD",
        "totalValueLockedUSD",
        "hourlyVolumeUSD",
        "hourlySupplySideRevenueUSD",
        "hourlyProtocolSideRevenueUSD",
        "hourlyTotalRevenueUSD",
        "hourlySwapCount",
    ]
    for col in numeric_cols:
        if col in snapshots_df.columns:
            snapshots_df[col] = pd.to_numeric(snapshots_df[col], errors="coerce")
    display(snapshots_df.head())
else:
    print(f"No liquidityPoolHourlySnapshots returned for pool {POOL_ADDRESS}")

# Persist the small sample for later inspection
try:
    snapshots_df.to_parquet(
        DATA_DIR / f"liquidityPoolHourlySnapshots_sample_{POOL_ADDRESS}_{SAMPLE_SIZE}.parquet",
        index=False,
    )
except Exception:
    pass


In [ ]:
# Download data from entry "swaps"

SWAPS_QUERY = gql("""
query($pool: String!, $first: Int!) {
    swaps(
        first: $first,
        orderBy: timestamp,
        orderDirection: desc,
        where: { pool: $pool }
    ) {
        id
        hash
        nonce
        logIndex
        gasLimit
        gasUsed
        gasPrice
        protocol {
            id
        }
        account {
            id
        }
        pool {
            id
        }
        blockNumber
        timestamp
        tick
        tokenIn {
            id
            symbol
        }
        amountIn
        amountInUSD
        tokenOut {
            id
            symbol
        }
        amountOut
        amountOutUSD
        reserveAmounts
    }
}
""")


swaps_resp = uniswap_v3_arbitrum_client.execute(
    SWAPS_QUERY,
    variable_values={"pool": POOL_ADDRESS, "first": SAMPLE_SIZE},
)
swaps_rows = swaps_resp.get("swaps", [])
swaps_df = pd.DataFrame(swaps_rows)

if not swaps_df.empty:
    swaps_df["timestamp"] = swaps_df["timestamp"].astype(int)
    swaps_df["datetime"] = pd.to_datetime(swaps_df["timestamp"], unit="s", utc=True)
    for col in ["amount0", "amount1", "amountUSD", "liquidity", "sqrtPriceX96", "tick"]:
        if col in swaps_df.columns:
            swaps_df[col] = pd.to_numeric(swaps_df[col], errors="coerce")
    display(swaps_df.head())
else:
    print(f"No swaps returned for pool {POOL_ADDRESS}")

try:
    swaps_df.to_parquet(
        DATA_DIR / f"swaps_sample_{POOL_ADDRESS}_{SAMPLE_SIZE}.parquet",
        index=False,
    )
except Exception:
    pass



### 4.2 Subgrounds

In [ ]:
from subgrounds import Subgrounds

sg = Subgrounds()
print("Subgrounds initialized")

In [ ]:
# Using Subgrounds to fetch swaps data for the specified pool
headers = {"Authorization": f"Bearer {GRAPH_API_KEY}"} if GRAPH_API_KEY else {}
if headers:
    sg = Subgrounds(headers=headers)
else:
    print("Warning: GRAPH_API_KEY is not set; The Graph gateway requires an Authorization header.")

subgraph_url = UNISWAP_V3_ARBITRUM_ENDPOINT
uniswap_v3_subgraph = sg.load_subgraph(subgraph_url)

swaps = uniswap_v3_subgraph.Query.swaps(
    first=SAMPLE_SIZE,
    orderBy=uniswap_v3_subgraph.Swap.timestamp,
    orderDirection="desc",
    where={"pool": POOL_ADDRESS},
)

swaps_fields = [
    swaps.id,
    swaps.hash,
    swaps.timestamp,
    swaps.tick,
    swaps.amountIn,
    swaps.amountOut,
    swaps.amountInUSD,
    swaps.amountOutUSD,
]

swaps_df_sg = sg.query_df(swaps_fields)

if not swaps_df_sg.empty:
    swaps_df_sg.columns = [col.replace("swaps_", "") for col in swaps_df_sg.columns]
    swaps_df_sg["timestamp"] = swaps_df_sg["timestamp"].astype(int)
    swaps_df_sg["datetime"] = pd.to_datetime(swaps_df_sg["timestamp"], unit="s", utc=True)
    for col in ["amountIn", "amountOut", "amountInUSD", "amountOutUSD", "tick"]:
        if col in swaps_df_sg.columns:
            swaps_df_sg[col] = pd.to_numeric(swaps_df_sg[col], errors="coerce")
    display(swaps_df_sg.head())
else:
    print(f"No swaps returned for pool {POOL_ADDRESS} using Subgrounds")


## 5. Data cleaning & feature engineering
Unify time indices and perform feature engineering.

In [ ]:
# Basic parsing
if not swaps_df.empty:
    swaps_df["timestamp"] = swaps_df["timestamp"].astype(int)
    swaps_df["datetime"] = pd.to_datetime(swaps_df["timestamp"], unit="s", utc=True)

    numeric_cols = [
        "amountIn",
        "amountOut",
        "amountInUSD",
        "amountOutUSD",
        "gasLimit",
        "gasUsed",
        "gasPrice",
        "tick",
        "blockNumber",
    ]
    for col in numeric_cols:
        if col in swaps_df.columns:
            swaps_df[col] = pd.to_numeric(swaps_df[col], errors="coerce")

    swaps_df = swaps_df.sort_values("datetime")

    # Price feature: tokenOut per tokenIn (handle zeros)
    swaps_df["price"] = swaps_df["amountOut"] / swaps_df["amountIn"].replace(0, np.nan)
    swaps_df["log_price"] = np.log(swaps_df["price"])

    # Volume and direction (use USD if present)
    usd_in = swaps_df.get("amountInUSD")
    usd_out = swaps_df.get("amountOutUSD")
    swaps_df["abs_amountUSD"] = (
        usd_in.abs() if usd_in is not None else pd.Series(index=swaps_df.index, dtype="float")
    )
    if usd_out is not None:
        swaps_df["abs_amountUSD"] = swaps_df["abs_amountUSD"].fillna(usd_out.abs())

    swaps_df["direction"] = np.sign(swaps_df["amountOut"].fillna(0))

    swaps_df = swaps_df.dropna(subset=["price", "log_price"])
    display(swaps_df.head())
else:
    print(f"No swaps returned for pool {POOL_ADDRESS}")

## 6. Compute liquidity and price metrics
Compute liquidity, TWAP, and other indicators.

In [ ]:
# TWAP (example: 5-minute window)
window = "5min"

if swaps_df is None or swaps_df.empty:
    print("swaps_df is empty; run the swaps download cell first.")
else:
    # Ensure datetime column exists
    if "datetime" not in swaps_df.columns:
        if "timestamp" in swaps_df.columns:
            swaps_df["timestamp"] = pd.to_numeric(swaps_df["timestamp"], errors="coerce")
            swaps_df["datetime"] = pd.to_datetime(swaps_df["timestamp"], unit="s", utc=True)
        elif swaps_df.index.name == "datetime":
            swaps_df = swaps_df.reset_index()
        else:
            raise KeyError("Missing datetime column; run the feature engineering cell first.")

    swaps_df = swaps_df.set_index("datetime")
    swaps_df["twap"] = swaps_df["log_price"].rolling(window).mean().pipe(np.exp)

    # Liquidity proxy from available fields (USD if present, else token amounts)
    if "abs_amountUSD" in swaps_df.columns and swaps_df["abs_amountUSD"].notna().any():
        liquidity_proxy = swaps_df["abs_amountUSD"]
    else:
        liquidity_proxy = swaps_df["amountOut"].abs().where(swaps_df["amountOut"].notna(), swaps_df["amountIn"].abs())

    liquidity_stats = liquidity_proxy.describe()
    display(liquidity_stats)

## 7. Fee and slippage estimation
Estimate fees and price impact from historical trades.

In [ ]:
FEE_TIER = float(os.getenv("FEE_TIER", "0.0005"))  # 0.05%

if swaps_df is None or swaps_df.empty:
    print("swaps_df is empty; run the swaps download cell first.")
else:
    # Use USD volume if available, else fallback to token amounts
    if "abs_amountUSD" in swaps_df.columns and swaps_df["abs_amountUSD"].notna().any():
        volume_usd = swaps_df["abs_amountUSD"]
    else:
        volume_usd = swaps_df["amountOut"].abs().where(swaps_df["amountOut"].notna(), swaps_df["amountIn"].abs())

    # Fee estimate: trade value * fee rate
    swaps_df["fee_usd"] = volume_usd * FEE_TIER

    # Simplified slippage proxy: trade value / liquidity proxy
    if "liquidity_proxy" in globals():
        base_liquidity = liquidity_proxy
    else:
        base_liquidity = volume_usd.replace(0, np.nan)
    swaps_df["slippage_proxy"] = volume_usd / base_liquidity

    swaps_df[["fee_usd", "slippage_proxy"]].describe()

    display(swaps_df[["fee_usd", "slippage_proxy"]].head())

## 8. Strategy backtest implementation
Example strategy: TWAP mean-reversion signal.

In [ ]:
if swaps_df is None or swaps_df.empty:
    print("swaps_df is empty; run the swaps download cell first.")
else:
    # Ensure TWAP exists
    if "twap" not in swaps_df.columns:
        swaps_df["twap"] = swaps_df["log_price"].rolling("5min").mean().pipe(np.exp)

    # Simple signal: price deviation from TWAP
    swaps_df["signal"] = (swaps_df["price"] - swaps_df["twap"]) / swaps_df["twap"]

    # Position: go contrarian when deviation is large
    threshold = float(os.getenv("SIGNAL_THRESHOLD", "0.002"))
    swaps_df["position"] = np.where(
        swaps_df["signal"] > threshold,
        -1,
        np.where(swaps_df["signal"] < -threshold, 1, 0),
    )

    # Simplified returns: next period price change * position
    swaps_df["price_return"] = swaps_df["price"].pct_change().shift(-1)
    swaps_df["strategy_return"] = swaps_df["position"] * swaps_df["price_return"]

    swaps_df[["strategy_return"]].dropna().describe()

    display(swaps_df[["strategy_return"]].head())

## 9. 结果可视化与评估
输出收益曲线、回撤与风险指标。

In [ ]:
import matplotlib.pyplot as plt

if swaps_df is None or swaps_df.empty or "strategy_return" not in swaps_df.columns:
    print("strategy_return is missing; run the strategy cell first.")
else:
    result = swaps_df[["strategy_return"]].dropna().copy()
    result["equity"] = (1 + result["strategy_return"]).cumprod()
    result["peak"] = result["equity"].cummax()
    result["drawdown"] = result["equity"] / result["peak"] - 1

    fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
    result["equity"].plot(ax=ax[0], title="Equity Curve")
    result["drawdown"].plot(ax=ax[1], title="Drawdown")
    plt.tight_layout()

    result[["equity", "drawdown"]].tail()

    display(result[["equity", "drawdown"]].tail())